In [ ]:
!pip install youtube-transcript-api

# GET GROUND TRUTH TRANSCRIPT

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from urllib.parse import urlparse, parse_qs

def get_youtube_video_id(url):
    """
    Extracts the video ID from various forms of YouTube URLs.
    """
    parsed_url = urlparse(url)

    if parsed_url.hostname in ('youtu.be', 'www.youtu.be'):
        return parsed_url.path[1:]

    if parsed_url.hostname in ('youtube.com', 'www.youtube.com'):
        if parsed_url.path == '/watch':
            return parse_qs(parsed_url.query).get('v', [None])[0]

    return None

def fetch_youtube_transcript(video_url):
    video_id = get_youtube_video_id(video_url)

    if not video_id:
        return "Error: Could not extract a valid Video ID from the provided URL."

    try:
        ytt_api = YouTubeTranscriptApi()

        # ADDED: languages=['hi', 'en']
        # This tells the API to try Hindi first, then English.
        fetched_transcript = ytt_api.fetch(video_id, languages=['hi', 'en'])

        transcript_data = fetched_transcript.to_raw_data()

        # Extract just the text and join it into a single string.
        full_transcript = " ".join([entry['text'] for entry in transcript_data])

        return full_transcript

    except Exception as e:
        return f"An error occurred while fetching the transcript:\n{e}"

# --- Example Usage ---
if __name__ == "__main__":
    # The video ID from your error message
    url = "INSERT YOUTUBE LINK"

    print(f"Fetching transcript for {url}...")
    transcript = fetch_youtube_transcript(url)

    print("\n--- Transcript ---")
    print(transcript)

# FIND THE GROUND TRUTH TRANSCRIPT LENGTH

In [ ]:
import re

def count_words_regex(text):

    words = text.split()


    print("Matched words:", words)

    return len(words)

N=count_words_regex(transcript)

print(f"Hindi word count (Regex): {count_words_regex(transcript)}")


INSERT ANNOTATED TRANSCRIPT TO FIND I+S+D

In [ ]:
import re

text = """ INSERT_ANNOTATED_TRANSCRIPT """

sub_pattern = r'\[SUBSTITUTI?ON[^\]]*\]' # Made the 'I' optional to catch the typo "SUBSTITUION"
ins_pattern = r'\[INSERTION[^\]]*\]'
del_pattern = r'\[DELETION[^\]]*\]'

# Extract all matches ignoring case
substitutions = re.findall(sub_pattern, text, flags=re.IGNORECASE)
insertions = re.findall(ins_pattern, text, flags=re.IGNORECASE)
deletions = re.findall(del_pattern, text, flags=re.IGNORECASE)

# Print the final counts
print("--- Error Annotation Counts ---")
print(f"Total SUBSTITUTIONS: {len(substitutions)}")
print(f"Total INSERTIONS:    {len(insertions)}")
print(f"Total DELETIONS:     {len(deletions)}")
print("-" * 31)
print(f"Total Annotations:   {len(substitutions) + len(insertions) + len(deletions)}")

In [ ]:
WER=(len(substitutions) + len(insertions) + len(deletions))/N*100